# Evaluate Census Foundation Models

This notebook rebuilds the same cross-validation evaluation used in `train_census_foundation_models.py` and adds confusion matrices plus imbalance-aware metrics.

Important: the training script only saved aggregate CSV metrics, not fitted model artifacts or per-row predictions. That means confusion matrices for the trained run cannot be recovered directly from `fold_metrics.csv` alone. This notebook reproduces the same folds and generates out-of-fold predictions so you can compare the models fairly.

In [26]:
from __future__ import annotations

import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from catboost import CatBoostClassifier
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from tabicl import TabICLClassifier
from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion

sns.set_theme(style="whitegrid")

## Paths

Set `PROJECT_ROOT` to the server checkout root that contains:
- `census-bureau.data`
- `census-bureau.columns`
- `cluster_tabular_models/outputs/census_foundation_models/resolved_config.json`

In [ ]:
PROJECT_ROOT = Path("/home/lhuang37/repos/homework")
OUTPUT_DIR = PROJECT_ROOT / "cluster_tabular_models/outputs/census_foundation_models"
CONFIG_PATH = OUTPUT_DIR / "resolved_config.json"
DATA_PATH = PROJECT_ROOT / "census-bureau.data"
COLUMNS_PATH = PROJECT_ROOT / "census-bureau.columns"

SHOW_PER_FOLD_CONFUSION = True
THRESHOLDS = {
    "catboost": 0.5,
    "tabpfn_v2_5": 0.5,
    "tabicl": 0.5,
}

for path in [PROJECT_ROOT, OUTPUT_DIR, CONFIG_PATH, DATA_PATH, COLUMNS_PATH]:
    print(path)
    print("  exists:", path.exists())

/home/lhuang37/repos/homework
  exists: True
/home/lhuang37/repos/homework/cluster_tabular_models/outputs/census_foundation_models
  exists: True
/home/lhuang37/repos/homework/cluster_tabular_models/outputs/census_foundation_models/resolved_config.json
  exists: True
/home/lhuang37/repos/homework/census-bureau.data
  exists: True
/home/lhuang37/repos/homework/census-bureau.columns
  exists: True


In [28]:
CATEGORICAL_CODES = [
    "detailed industry recode",
    "detailed occupation recode",
    "own business or self employed",
    "veterans benefits",
    "year",
]


def load_config(path: Path) -> dict:
    return json.loads(path.read_text())


def fill_missing_values(x: pd.DataFrame, numeric_columns: list[str], categorical_columns: list[str]) -> pd.DataFrame:
    filled = x.copy()

    for column in numeric_columns:
        if filled[column].isna().any():
            filled[column] = filled[column].fillna(filled[column].median())

    for column in categorical_columns:
        if filled[column].isna().any():
            modes = filled[column].mode(dropna=True)
            fill_value = modes.iloc[0] if not modes.empty else "Missing"
            filled[column] = filled[column].fillna(fill_value).astype(object)

    return filled


def load_dataset(target_column: str):
    columns = [line.strip() for line in COLUMNS_PATH.read_text().splitlines() if line.strip()]
    df = pd.read_csv(DATA_PATH, header=None, names=columns, na_values="?", skipinitialspace=True)

    object_columns = df.select_dtypes(include=["object", "string"]).columns
    for column in object_columns:
        df[column] = df[column].map(lambda value: value.strip() if isinstance(value, str) else value).astype(object)

    x = df.drop(columns=[target_column]).copy()
    x_object_columns = x.select_dtypes(include=["object", "string"]).columns
    for column in x_object_columns:
        x[column] = x[column].where(pd.notna(x[column]), np.nan).astype(object)

    y = df[target_column].map(lambda value: value.strip() if isinstance(value, str) else value).astype(str)

    categorical_columns = list(x.select_dtypes(include=["object", "string"]).columns)
    categorical_columns.extend(column for column in CATEGORICAL_CODES if column in x.columns)
    categorical_columns = list(dict.fromkeys(categorical_columns))
    numeric_columns = [column for column in x.columns if column not in categorical_columns]
    x = fill_missing_values(x, numeric_columns, categorical_columns)
    return x, y, numeric_columns, categorical_columns


def build_dense_preprocessor(numeric_columns: list[str], categorical_columns: list[str]) -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),
                numeric_columns,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
                        (
                            "encoder",
                            OrdinalEncoder(
                                handle_unknown="use_encoded_value",
                                unknown_value=-1,
                                encoded_missing_value=-1,
                            ),
                        ),
                    ]
                ),
                categorical_columns,
            ),
        ]
    )


def build_foundation_preprocessor(numeric_columns: list[str], categorical_columns: list[str]) -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            ("num", "passthrough", numeric_columns),
            (
                "cat",
                OrdinalEncoder(
                    handle_unknown="use_encoded_value",
                    unknown_value=-1,
                    encoded_missing_value=-1,
                ),
                categorical_columns,
            ),
        ]
    )


def prepare_catboost_frames(x_train: pd.DataFrame, x_test: pd.DataFrame, categorical_columns: list[str]):
    train_frame = x_train.copy()
    test_frame = x_test.copy()

    for column in categorical_columns:
        train_frame[column] = train_frame[column].fillna("Missing").astype(str)
        test_frame[column] = test_frame[column].fillna("Missing").astype(str)

    return train_frame, test_frame


def subsample_training_rows(x_train: pd.DataFrame, y_train: pd.Series, limit: int | None, seed: int):
    if limit is None or len(x_train) <= limit:
        return x_train, y_train

    class_counts = y_train.value_counts().sort_index()
    class_limits = {}
    remaining = limit

    for class_value, class_count in class_counts.items():
        allocated = max(1, int(round(limit * class_count / len(y_train))))
        allocated = min(allocated, int(class_count))
        class_limits[class_value] = allocated
        remaining -= allocated

    class_order = list(class_counts.sort_values(ascending=False).index)
    cursor = 0
    while remaining != 0 and class_order:
        class_value = class_order[cursor % len(class_order)]
        max_for_class = int(class_counts[class_value])

        if remaining > 0 and class_limits[class_value] < max_for_class:
            class_limits[class_value] += 1
            remaining -= 1
        elif remaining < 0 and class_limits[class_value] > 1:
            class_limits[class_value] -= 1
            remaining += 1

        cursor += 1

    grouped_indices = []
    for class_value, class_index in y_train.groupby(y_train).groups.items():
        class_rows = x_train.loc[class_index]
        sampled_rows = class_rows.sample(
            n=class_limits[int(class_value)],
            random_state=seed,
        )
        grouped_indices.extend(sampled_rows.index.tolist())

    sampled_index = pd.Index(grouped_indices).to_series().sample(frac=1.0, random_state=seed).to_list()
    sampled_x = x_train.loc[sampled_index].reset_index(drop=True)
    sampled_y = y_train.loc[sampled_index].reset_index(drop=True).astype(int)
    return sampled_x, sampled_y


def predict_scores(model, x_test, batch_size: int) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        chunks = []
        for start in range(0, len(x_test), batch_size):
            stop = start + batch_size
            probabilities = model.predict_proba(x_test[start:stop])
            probabilities = np.asarray(probabilities)
            if probabilities.ndim == 1:
                chunks.append(probabilities.astype(float))
            else:
                chunks.append(probabilities[:, 1].astype(float))
        return np.concatenate(chunks)

    predictions = model.predict(x_test)
    return np.asarray(predictions, dtype=float)


def build_model(model_name: str, model_params: dict, seed: int):
    if model_name == "catboost":
        params = {
            "random_seed": seed,
            "allow_writing_files": False,
            **model_params,
        }
        return CatBoostClassifier(**params)

    if model_name == "tabpfn_v2_5":
        return TabPFNClassifier.create_default_for_version(ModelVersion.V2_5, **model_params)

    if model_name == "tabicl":
        return TabICLClassifier(**model_params)

    raise ValueError(f"Unsupported model: {model_name}")

In [29]:
def evaluate_models_with_oof_predictions(config: dict):
    x, y_labels, numeric_columns, categorical_columns = load_dataset(config["target_column"])
    y = y_labels.eq(config["positive_label"]).astype(int)

    splitter = StratifiedKFold(
        n_splits=config["n_splits"],
        shuffle=True,
        random_state=config["seed"],
    )

    dense_preprocessor = build_dense_preprocessor(numeric_columns, categorical_columns)
    foundation_preprocessor = build_foundation_preprocessor(numeric_columns, categorical_columns)

    fold_rows = []
    prediction_rows = []

    for fold_index, (train_idx, test_idx) in enumerate(splitter.split(x, y), start=1):
        x_train_raw = x.iloc[train_idx].reset_index(drop=True)
        x_test_raw = x.iloc[test_idx].reset_index(drop=True)
        y_train = y.iloc[train_idx].reset_index(drop=True)
        y_test = y.iloc[test_idx].reset_index(drop=True)

        x_train_dense = dense_preprocessor.fit_transform(x_train_raw)
        x_test_dense = dense_preprocessor.transform(x_test_raw)
        x_train_foundation = foundation_preprocessor.fit_transform(x_train_raw)
        x_test_foundation = foundation_preprocessor.transform(x_test_raw)

        for model_config in config["models"]:
            model_name = model_config["name"]
            train_limit = model_config.get("train_row_limit")
            predict_batch_size = model_config.get("predict_batch_size", len(x_test_raw))

            if model_name == "catboost":
                x_train_model, y_train_model = subsample_training_rows(
                    x_train_raw,
                    y_train,
                    train_limit,
                    seed=config["seed"] + fold_index,
                )
                x_test_model = x_test_raw
                x_train_model, x_test_model = prepare_catboost_frames(
                    x_train_model,
                    x_test_model,
                    categorical_columns,
                )
                cat_features = [x_train_model.columns.get_loc(column) for column in categorical_columns]
            else:
                if model_name == "tabpfn_v2_5":
                    train_frame = pd.DataFrame(x_train_foundation)
                    test_frame = pd.DataFrame(x_test_foundation)
                else:
                    train_frame = pd.DataFrame(x_train_dense)
                    test_frame = pd.DataFrame(x_test_dense)

                x_train_model, y_train_model = subsample_training_rows(
                    train_frame,
                    y_train,
                    train_limit,
                    seed=config["seed"] + fold_index,
                )
                x_test_model = test_frame
                cat_features = None

            model = build_model(model_name, model_config.get("params", {}), seed=config["seed"] + fold_index)

            fit_started = time.perf_counter()
            if model_name == "catboost":
                model.fit(x_train_model, y_train_model, cat_features=cat_features)
            else:
                model.fit(x_train_model, y_train_model)
            fit_seconds = time.perf_counter() - fit_started

            predict_started = time.perf_counter()
            scores = predict_scores(model, x_test_model, predict_batch_size)
            predict_seconds = time.perf_counter() - predict_started

            threshold = THRESHOLDS.get(model_name, 0.5)
            predictions = (scores >= threshold).astype(int)
            tn, fp, fn, tp = confusion_matrix(y_test, predictions).ravel()

            fold_rows.append(
                {
                    "fold": fold_index,
                    "model": model_name,
                    "threshold": threshold,
                    "train_rows": len(x_train_model),
                    "test_rows": len(x_test_model),
                    "positive_rate": float(y_test.mean()),
                    "roc_auc": roc_auc_score(y_test, scores),
                    "average_precision": average_precision_score(y_test, scores),
                    "accuracy": accuracy_score(y_test, predictions),
                    "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
                    "precision": precision_score(y_test, predictions, zero_division=0),
                    "recall": recall_score(y_test, predictions, zero_division=0),
                    "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
                    "f1": f1_score(y_test, predictions, zero_division=0),
                    "tn": int(tn),
                    "fp": int(fp),
                    "fn": int(fn),
                    "tp": int(tp),
                    "fit_seconds": fit_seconds,
                    "predict_seconds": predict_seconds,
                }
            )

            prediction_rows.append(
                pd.DataFrame(
                    {
                        "fold": fold_index,
                        "model": model_name,
                        "y_true": y_test.to_numpy(),
                        "score": scores,
                        "threshold": threshold,
                        "y_pred": predictions,
                    }
                )
            )

    fold_metrics = pd.DataFrame(fold_rows)
    oof_predictions = pd.concat(prediction_rows, ignore_index=True)
    return fold_metrics, oof_predictions


def summarize_oof_predictions(oof_predictions: pd.DataFrame) -> pd.DataFrame:
    summary_rows = []

    for model_name, model_frame in oof_predictions.groupby("model"):
        y_true = model_frame["y_true"].to_numpy()
        y_pred = model_frame["y_pred"].to_numpy()
        scores = model_frame["score"].to_numpy()
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        summary_rows.append(
            {
                "model": model_name,
                "threshold": float(model_frame["threshold"].iloc[0]),
                "rows": len(model_frame),
                "positive_rate": float(np.mean(y_true)),
                "roc_auc": roc_auc_score(y_true, scores),
                "average_precision": average_precision_score(y_true, scores),
                "accuracy": accuracy_score(y_true, y_pred),
                "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
                "precision": precision_score(y_true, y_pred, zero_division=0),
                "recall": recall_score(y_true, y_pred, zero_division=0),
                "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
                "f1": f1_score(y_true, y_pred, zero_division=0),
                "tn": int(tn),
                "fp": int(fp),
                "fn": int(fn),
                "tp": int(tp),
            }
        )

    return pd.DataFrame(summary_rows).sort_values("average_precision", ascending=False).reset_index(drop=True)

In [30]:
config = load_config(CONFIG_PATH)
print(json.dumps(config, indent=2))

existing_fold_metrics_path = OUTPUT_DIR / "fold_metrics.csv"
existing_summary_metrics_path = OUTPUT_DIR / "summary_metrics.csv"

if existing_fold_metrics_path.exists():
    print("\Existing saved fold metrics from training:")
    display(pd.read_csv(existing_fold_metrics_path).head())

if existing_summary_metrics_path.exists():
    print("\Existing saved summary metrics from training:")
    display(pd.read_csv(existing_summary_metrics_path))

{
  "seed": 42,
  "n_splits": 5,
  "target_column": "label",
  "positive_label": "50000+.",
  "output_dir": "cluster_tabular_models/outputs/census_foundation_models",
  "models": [
    {
      "name": "catboost",
      "train_row_limit": null,
      "predict_batch_size": 50000,
      "params": {
        "iterations": 1500,
        "learning_rate": 0.05,
        "depth": 8,
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "task_type": "GPU",
        "verbose": false
      }
    },
    {
      "name": "tabpfn_v2_5",
      "train_row_limit": 50000,
      "predict_batch_size": 1024,
      "params": {}
    },
    {
      "name": "tabicl",
      "train_row_limit": 100000,
      "predict_batch_size": 2048,
      "params": {}
    }
  ]
}
\Existing saved fold metrics from training:


,fold,model,train_rows,test_rows,roc_auc,average_precision,accuracy,f1,fit_seconds,predict_seconds
0,1,catboost,159618,39905,0.953099,0.685419,0.958226,0.593117,63.823366,0.453458
1,1,tabpfn_v2_5,50000,39905,0.948293,0.650650,0.955444,0.549645,13.010627,2014.283036
2,1,tabicl,100000,39905,0.948499,0.649220,0.954943,0.546646,6.040606,502.569821
3,2,catboost,159618,39905,0.954763,0.683677,0.958201,0.588555,71.772053,0.358043
4,2,tabpfn_v2_5,50000,39905,0.949800,0.650731,0.955294,0.545825,12.231924,2005.314905


\Existing saved summary metrics from training:


,model,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std,accuracy_mean,accuracy_std,f1_mean,f1_std,fit_seconds_mean,fit_seconds_std,predict_seconds_mean,predict_seconds_std
0,catboost,0.954816,0.001474,0.689180,0.005453,0.958411,0.000535,0.589244,0.004260,79.406571,23.292893,0.429717,0.073078
1,tabicl,0.950628,0.001876,0.653899,0.004472,0.955554,0.000446,0.547820,0.003044,3.724477,1.337239,502.964260,0.758282
2,tabpfn_v2_5,0.949966,0.001770,0.655108,0.005584,0.955775,0.000486,0.548975,0.002377,12.537696,1.244259,2009.997164,4.498047
